In [1]:
%config InlineBackend.figure_formats = ['svg']

In [2]:
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['FreeSans']

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
state_dim = 64
num_actions = 25
horizon = 20

In [5]:
import sys
sys.path.insert(0, '../4_BCQf')
from model import BCQf, all_subactions_vec
from data import EpisodicBuffer as EpisodicBufferFF
from data import remap_rewards

In [6]:
from evaluate import (
    EpisodicBufferO, offline_evaluation_O,
    EpisodicBufferF, offline_evaluation_F,
)

In [7]:
from types import SimpleNamespace
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
from joblib import Parallel, delayed

In [8]:
import pandas as pd, numpy as np, re
from tqdm import tqdm

# Load test data
test_episodes = EpisodicBufferF(state_dim, num_actions, horizon)
test_episodes.load('../data/episodes+encoded_state+knn_pibs_factored/test_data.pt')
test_episodes.reward = remap_rewards(
    test_episodes.reward,
    SimpleNamespace(**{'R_immed': 0.0, 'R_death': 0.0, 'R_disch': 100.0})
)
loader = DataLoader(test_episodes, batch_size=len(test_episodes), shuffle=False)
test_batch_O = next(iter(loader))

logdir = '../4_BCQf/logs/mimic_dBCQf'
top_ess_list = []
for ver in range(40):
    text = open(f'{logdir}/version_{ver}/hparams.yaml').read()
    thresh = float(re.search(r'threshold: ([\d.]+)', text).group(1))
    seed = int(re.search(r'seed: (\d+)', text).group(1))
    df = pd.read_csv(f'{logdir}/version_{ver}/metrics.csv')
    valid = df.dropna(subset=['val_wis', 'val_ess'])
    if len(valid) > 0:
        top = valid.loc[valid['val_ess'].idxmax()]
        top_ess_list.append({
            'version': ver, 'threshold': thresh, 'seed': seed,
            'val_wis': top['val_wis'], 'val_ess': top['val_ess'],
            'iteration': top['iteration'],
        })

top20 = pd.DataFrame(top_ess_list).nlargest(20, 'val_ess')
print('Top 20 by val_ess:')
print(top20[['version', 'threshold', 'seed', 'val_ess', 'val_wis']].to_string(index=False))

top20_results = []
for _, row in tqdm(top20.iterrows(), total=len(top20)):
    ver = int(row['version'])
    model = BCQf.load_from_checkpoint(
        f'{logdir}/version_{ver}/step=10000.ckpt',
        map_location='cpu', weights_only=False
    )
    model.eval()
    model.all_subactions_vec = all_subactions_vec
    w, e = offline_evaluation_F(model, test_batch_O, weighted=True, eps=0.01)
    top20_results.append({
        'threshold': row['threshold'], 'seed': row['seed'],
        'test_wis': w, 'test_ess': e,
    })

df_top20 = pd.DataFrame(top20_results)
print('\nTop 20 test performance:')
for _, r in df_top20.iterrows():
    print(f"  thresh={r['threshold']:.4f}  seed={int(r['seed'])}  WIS={r['test_wis']:.2f}  ESS={r['test_ess']:.1f}")

Episodic Buffer loaded with 2757 episides.
Top 20 by val_ess:
 version  threshold  seed    val_ess   val_wis
      33     0.7500     3 236.145920 93.837357
      25     0.5000     0 233.521881 93.284309
      38     0.9999     3 231.073273 93.339951
      31     0.7500     1 230.953217 93.905128
      32     0.7500     2 230.170364 93.549095
      28     0.5000     3 229.701996 94.238365
      30     0.7500     0 228.469360 92.951118
      27     0.5000     2 228.355621 93.849480
      34     0.7500     4 228.289200 93.113762
      39     0.9999     4 227.400391 93.807083
      36     0.9999     1 226.855164 94.471733
      29     0.5000     4 226.668884 93.542168
      35     0.9999     0 226.036972 93.445648
      26     0.5000     1 225.919662 93.486717
      37     0.9999     2 225.540802 94.478134
      23     0.3000     3 218.408997 95.001495
      22     0.3000     2 217.576447 95.413612
      21     0.3000     1 216.598801 95.059570
      20     0.3000     0 215.845261 93.78012

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:19<00:00,  1.05it/s]


Top 20 test performance:
  thresh=0.7500  seed=3  WIS=93.90  ESS=132.7
  thresh=0.5000  seed=0  WIS=93.36  ESS=102.1
  thresh=0.9999  seed=3  WIS=94.09  ESS=136.6
  thresh=0.7500  seed=1  WIS=94.36  ESS=121.4
  thresh=0.7500  seed=2  WIS=94.05  ESS=120.1
  thresh=0.5000  seed=3  WIS=94.95  ESS=113.0
  thresh=0.7500  seed=0  WIS=93.67  ESS=126.6
  thresh=0.5000  seed=2  WIS=93.35  ESS=85.9
  thresh=0.7500  seed=4  WIS=94.38  ESS=127.9
  thresh=0.9999  seed=4  WIS=93.92  ESS=133.5
  thresh=0.9999  seed=1  WIS=93.92  ESS=133.2
  thresh=0.5000  seed=4  WIS=93.88  ESS=93.6
  thresh=0.9999  seed=0  WIS=93.93  ESS=133.7
  thresh=0.5000  seed=1  WIS=92.32  ESS=88.4
  thresh=0.9999  seed=2  WIS=93.85  ESS=131.7
  thresh=0.3000  seed=3  WIS=95.73  ESS=135.3
  thresh=0.3000  seed=2  WIS=95.89  ESS=116.5
  thresh=0.3000  seed=1  WIS=95.54  ESS=131.2
  thresh=0.3000  seed=0  WIS=96.34  ESS=128.2
  thresh=0.3000  seed=4  WIS=95.45  ESS=128.5


In [9]:
# df_1_best = pd.read_csv('best_BCQ_meta.csv')
df_2_best = pd.read_csv('best_BCQf_meta.csv')

In [10]:
df_2_best

,version,threshold,seed,val_wis,val_ess,iteration
0,20,0.3,0,93.78,215.85,6368


In [11]:
# BCQ baseline not available — load only BCQf (non-non-shifted)
model_2 = BCQf.load_from_checkpoint(
    checkpoint_path=f'../4_BCQf/logs/mimic_dBCQf/version_{df_2_best["version"].item()}/step={(int(df_2_best["iteration"].item())//100)*100}.ckpt',
    map_location='cpu',
    weights_only=False
)
model_2.eval()
model_2.all_subactions_vec = all_subactions_vec

In [12]:
test_episodes_O = EpisodicBufferF(state_dim, num_actions, horizon)
test_episodes_O.load('../data/episodes+encoded_state+knn_pibs_factored/test_data.pt')
test_episodes_O.reward = remap_rewards(test_episodes_O.reward, SimpleNamespace(**{'R_immed': 0.0, 'R_death': 0.0, 'R_disch': 100.0}))

tmp_test_episodes_loader_O = DataLoader(test_episodes_O, batch_size=len(test_episodes_O), shuffle=False)
test_batch_O = next(iter(tmp_test_episodes_loader_O))

Episodic Buffer loaded with 2757 episides.


In [13]:
# test_wis_1, test_ess_1 = model_1.offline_evaluation(test_batch_O, weighted=True, eps=0.01)
test_wis_2, test_ess_2 = offline_evaluation_F(model_2, test_batch_O, weighted=True, eps=0.01)

In [14]:
print(f'Observed Test \t WIS: {test_episodes_O.reward.sum(axis=1).mean():.2f} \t ESS: {test_episodes_O.reward.shape[0]:.2f}')
# print(f'Baseline BCQ \t WIS: {test_wis_1:.2f} \t ESS: {test_ess_1:.2f}')
print(f'Factored BCQ \t WIS: {test_wis_2:.2f} \t ESS: {test_ess_2:.2f}')

Observed Test 	 WIS: 93.91 	 ESS: 2757.00
Factored BCQ 	 WIS: 96.34 	 ESS: 128.20


In [15]:
def bootstrap_test(i, buffer):
    n_episodes = len(buffer)
    idx = np.random.default_rng(seed=i).choice(n_episodes, n_episodes, replace=True)
    batch = buffer[idx]
    return batch[2].sum(axis=1).float().mean()

In [16]:
def bootstrap_WIS_2(i, buffer):
    n_episodes = len(buffer)
    idx = np.random.default_rng(seed=i).choice(n_episodes, n_episodes, replace=True)
    batch = buffer[idx]
    return offline_evaluation_F(model_2, batch, weighted=True, eps=0.01)

## 100 bootstraps

In [17]:
boot_test_reward = Parallel(n_jobs=6)(delayed(bootstrap_test)(i, test_episodes_O) for i in tqdm(range(100)))

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 39.02it/s]


In [18]:
# boot_test_wis_1, boot_test_ess_1 = zip(*Parallel(n_jobs=6)(delayed(bootstrap_WIS_1)(i, test_episodes_O) for i in tqdm(range(100))))

In [19]:
boot_test_wis_2, boot_test_ess_2 = zip(*Parallel(n_jobs=6)(delayed(bootstrap_WIS_2)(i, test_episodes_O) for i in tqdm(range(100))))

 12%|███████████████▎                                                                                                                | 12/100 [00:03<00:26,  3.38it/s]/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/PRISM-1.0/prism-venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:27<00:00,  3.57it/s]


In [20]:
print(f'Observed Test \t ' + 
      f'WIS: {test_episodes_O.reward.sum(axis=1).mean():.2f} ({np.quantile(boot_test_reward, 0.025):.2f}, {np.quantile(boot_test_reward, 0.975):.2f}) \t ' +
      f'ESS: {test_episodes_O.reward.shape[0]:.2f}')
print(f'Factored BCQ \t ' + 
      f'WIS: {test_wis_2:.2f} ({np.quantile(boot_test_wis_2, 0.025):.2f}, {np.quantile(boot_test_wis_2, 0.975):.2f}) \t ' +
      f'ESS: {test_ess_2:.2f} ({np.quantile(boot_test_ess_2, 0.025):.2f}, {np.quantile(boot_test_ess_2, 0.975):.2f})')

Observed Test 	 WIS: 93.91 (7.69, 8.28) 	 ESS: 2757.00
Factored BCQ 	 WIS: 96.34 (92.95, 99.12) 	 ESS: 128.20 (106.23, 148.27)


In [21]:
print(f'Observed Test \t ' + 
      f'WIS: {test_episodes_O.reward.sum(axis=1).mean():.2f} ± {np.std(boot_test_reward):.2f} \t ' +
      f'ESS: {test_episodes_O.reward.shape[0]:.2f}')
print(f'Factored BCQ \t ' + 
      f'WIS: {test_wis_2:.2f} ± {np.std(boot_test_wis_2):.2f} \t ' +
      f'ESS: {test_ess_2:.2f} ± {np.std(boot_test_ess_2):.2f}')

Observed Test 	 WIS: 93.91 ± 0.16 	 ESS: 2757.00
Factored BCQ 	 WIS: 96.34 ± 1.63 	 ESS: 128.20 ± 10.38
